In [ ]:
import sys, site
from pathlib import Path

################################# NOTE #################################
#  Please be aware that if colab installs the latest numpy and pyqlib  #
#  in this cell, users should RESTART the runtime in order to run the  #
#  following cells successfully.                  #
########################################################################

try:
    import qlib
except ImportError:
    # install qlib
    ! pip install --upgrade numpy
    ! pip install pyqlib
    if "google.colab" in sys.modules:
        # The Google colab environment is a little outdated. We have to downgrade the pyyaml to make it compatible with other packages
        ! pip install pyyaml==5.4.1
    # reload
    site.main()

scripts_dir = Path.cwd().parent.joinpath("scripts")
if not scripts_dir.joinpath("get_data.py").exists():
    # download get_data.py script
    scripts_dir = Path("~/tmp/qlib_code/scripts").expanduser().resolve()
    scripts_dir.mkdir(parents=True, exist_ok=True)
    import requests

    with requests.get("https://raw.githubusercontent.com/microsoft/qlib/main/scripts/get_data.py", timeout=10) as resp:
        with open(scripts_dir.joinpath("get_data.py"), "wb") as fp:
            fp.write(resp.content)

In [ ]:
import qlib
import pandas as pd
from qlib.constant import REG_CN
from qlib.utils import exists_qlib_data, init_instance_by_config
from qlib.workflow import R
from qlib.workflow.record_temp import SignalRecord, PortAnaRecord
from qlib.utils import flatten_dict

In [ ]:
# use default data
# NOTE: need to download data from remote: python scripts/get_data.py qlib_data_cn --target_dir ~/.qlib/qlib_data/cn_data
provider_uri = "~/.qlib/qlib_data/cn_data"  # target_dir
if not exists_qlib_data(provider_uri):
    print(f"Qlib data is not found in {provider_uri}")
    sys.path.append(str(scripts_dir))
    from get_data import GetData

    GetData().qlib_data(target_dir=provider_uri, region=REG_CN)
qlib.init(provider_uri=provider_uri, region=REG_CN)

Qlib data is not found in ~/.qlib/qlib_data/cn_data


2025-08-18 03:03:57.047 | WARNING  | qlib.tests.data:download:62 - The data for the example is collected from Yahoo Finance. Please be aware that the quality of the data might not be perfect. (You can refer to the original data source: https://finance.yahoo.com/lookup.)
2025-08-18 03:03:57.048 | INFO     | qlib.tests.data:download:65 - 20250818030356_qlib_data_cn_1d_latest.zip downloading......
196549632it [00:04, 48314757.40it/s]                               
2025-08-18 03:04:01.140 | WARNING  | qlib.tests.data:_unzip:124 - will delete the old qlib data directory(features, instruments, calendars, features_cache, dataset_cache): /root/.qlib/qlib_data/cn_data
2025-08-18 03:04:01.142 | INFO     | qlib.tests.data:_unzip:128 - /root/.qlib/qlib_data/cn_data/20250818030356_qlib_data_cn_1d_latest.zip unzipping......
100%|██████████| 31008/31008 [00:10<00:00, 3092.64it/s]
[7865:MainThread](2025-08-18 03:04:11,468) INFO - qlib.Initialization - [config.py:452] - default_conf: client.
[7865:Main

In [ ]:
market = "csi300"
benchmark = "SH000300"

# 训练

In [ ]:
###################################
# train model
###################################
data_handler_config = {
    "start_time": "2008-01-01",
    "end_time": "2020-08-01",
    "fit_start_time": "2008-01-01",
    "fit_end_time": "2014-12-31",
    "instruments": market,
}

task = {
    "model": {
        "class": "LGBModel",
        "module_path": "qlib.contrib.model.gbdt",
        "kwargs": {
            "loss": "mse",
            "colsample_bytree": 0.8879,
            "learning_rate": 0.0421,
            "subsample": 0.8789,
            "lambda_l1": 205.6999,
            "lambda_l2": 580.9768,
            "max_depth": 8,
            "num_leaves": 210,
            "num_threads": 20,
        },
    },
    "dataset": {
        "class": "DatasetH",
        "module_path": "qlib.data.dataset",
        "kwargs": {
            "handler": {
                "class": "Alpha158",
                "module_path": "qlib.contrib.data.handler",
                "kwargs": data_handler_config,
            },
            "segments": {
                "train": ("2008-01-01", "2014-12-31"),
                "valid": ("2015-01-01", "2016-12-31"),
                "test": ("2017-01-01", "2020-08-01"),
            },
        },
    },
}

# model initialization
model = init_instance_by_config(task["model"])
dataset = init_instance_by_config(task["dataset"])

# start exp to train model
with R.start(experiment_name="train_model"):
    R.log_params(**flatten_dict(task))
    model.fit(dataset)
    R.save_objects(trained_model=model)
    rid = R.get_recorder().id

[7865:MainThread](2025-08-18 03:08:51,054) INFO - qlib.timer - [log.py:127] - Time cost: 190.582s | Loading data Done
[7865:MainThread](2025-08-18 03:08:52,049) INFO - qlib.timer - [log.py:127] - Time cost: 0.329s | DropnaLabel Done
[7865:MainThread](2025-08-18 03:08:55,382) INFO - qlib.timer - [log.py:127] - Time cost: 3.332s | CSZScoreNorm Done
[7865:MainThread](2025-08-18 03:08:55,386) INFO - qlib.timer - [log.py:127] - Time cost: 4.330s | fit & process data Done
[7865:MainThread](2025-08-18 03:08:55,387) INFO - qlib.timer - [log.py:127] - Time cost: 194.916s | Init data Done
[7865:MainThread](2025-08-18 03:08:55,423) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name train_model.
[7865:MainThread](2025-08-18 03:08:55,430) INFO - qlib.workflow - [exp.py:258] - Experiment 219307569006216670 starts running ...
[7865:MainThread](2025-08-18 03:08:55,675) INFO - qlib.workflow - [recorder.py:345] - Recorder a03828df558141e8a492bc5821d7db

Training until validation scores don't improve for 50 rounds
[20]	train's l2: 0.990585	valid's l2: 0.99431
[40]	train's l2: 0.986931	valid's l2: 0.993693
[60]	train's l2: 0.984352	valid's l2: 0.99349
[80]	train's l2: 0.982319	valid's l2: 0.993382
[100]	train's l2: 0.980442	valid's l2: 0.99331
[120]	train's l2: 0.97871	valid's l2: 0.993247
[140]	train's l2: 0.976987	valid's l2: 0.993334
[160]	train's l2: 0.97536	valid's l2: 0.993338
Early stopping, best iteration is:
[122]	train's l2: 0.978519	valid's l2: 0.993238


[7865:MainThread](2025-08-18 03:10:40,991) INFO - qlib.timer - [log.py:127] - Time cost: 0.181s | waiting `async_log` Done


# 预测、回测、分析

In [ ]:
###################################
# prediction, backtest & analysis
###################################
port_analysis_config = {
    "executor": {
        "class": "SimulatorExecutor",
        "module_path": "qlib.backtest.executor",
        "kwargs": {
            "time_per_step": "day",
            "generate_portfolio_metrics": True,
        },
    },
    "strategy": {
        "class": "TopkDropoutStrategy",
        "module_path": "qlib.contrib.strategy.signal_strategy",
        "kwargs": {
            "model": model,
            "dataset": dataset,
            "topk": 50,
            "n_drop": 5,
        },
    },
    "backtest": {
        "start_time": "2017-01-01",
        "end_time": "2020-08-01",
        "account": 100000000,
        "benchmark": benchmark,
        "exchange_kwargs": {
            "freq": "day",
            "limit_threshold": 0.095,
            "deal_price": "close",
            "open_cost": 0.0005,
            "close_cost": 0.0015,
            "min_cost": 5,
        },
    },
}

# backtest and analysis
with R.start(experiment_name="backtest_analysis"):
    recorder = R.get_recorder(recorder_id=rid, experiment_name="train_model")
    model = recorder.load_object("trained_model")

    # prediction
    recorder = R.get_recorder()
    ba_rid = recorder.id
    sr = SignalRecord(model, dataset, recorder)
    sr.generate()

    # backtest & analysis
    par = PortAnaRecord(recorder, port_analysis_config, "day")
    par.generate()

[7865:MainThread](2025-08-18 03:14:17,761) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name backtest_analysis.
[7865:MainThread](2025-08-18 03:14:17,767) INFO - qlib.workflow - [exp.py:258] - Experiment 279414948458319968 starts running ...
[7865:MainThread](2025-08-18 03:14:17,778) INFO - qlib.workflow - [recorder.py:345] - Recorder 13611d0507e34998a4a8054510429a96 starts running under Experiment 279414948458319968 ...
[7865:MainThread](2025-08-18 03:14:17,791) INFO - qlib.workflow - [recorder.py:378] - Fail to log the uncommitted code of $CWD(/content) when run git diff.
[7865:MainThread](2025-08-18 03:14:17,801) INFO - qlib.workflow - [recorder.py:378] - Fail to log the uncommitted code of $CWD(/content) when run git status.
[7865:MainThread](2025-08-18 03:14:17,811) INFO - qlib.workflow - [recorder.py:378] - Fail to log the uncommitted code of $CWD(/content) when run git diff --cached.


[7865:MainThread](2025-08-18 03:14:22,932) INFO - qlib.workflow - [record_temp.py:198] - Signal record 'pred.pkl' has been saved as the artifact of the Experiment 279414948458319968


'The following are prediction results of the LGBModel model.'
                          score
datetime   instrument          
2017-01-03 SH600000   -0.042865
           SH600008    0.005925
           SH600009    0.030596
           SH600010   -0.013973
           SH600015   -0.141758


[7865:MainThread](2025-08-18 03:14:23,081) INFO - qlib.backtest caller - [__init__.py:93] - Create new exchange
[7865:MainThread](2025-08-18 03:14:50,275) WARNING - qlib.online operator - [exchange.py:219] - $close field data contains nan.
[7865:MainThread](2025-08-18 03:14:50,281) WARNING - qlib.online operator - [exchange.py:219] - $close field data contains nan.
[7865:MainThread](2025-08-18 03:14:50,293) WARNING - qlib.online operator - [exchange.py:226] - factor.day.bin file not exists or factor contains `nan`. Order using adjusted_price.
[7865:MainThread](2025-08-18 03:14:50,295) WARNING - qlib.online operator - [exchange.py:228] - trade unit 100 is not supported in adjusted_price mode.
/usr/local/lib/python3.11/dist-packages/qlib/contrib/strategy/signal_strategy.py:61: DeprecationWarning: `model` `dataset` is deprecated; use `signal`.
  warnings.warn("`model` `dataset` is deprecated; use `signal`.", DeprecationWarning)
[7865:MainThread](2025-08-18 03:15:21,332) WARNING - qlib.dat

backtest loop:   0%|          | 0/871 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/usr/local/lib/python3.11/dist-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/usr/local/lib/python3.11/dist-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
[7865:MainThread](2025-08-18 03:15:41,685) INFO - qlib.workflow - [record_temp.py:515] - Portfolio analysis record 'port_analysis_1day.pkl' has been saved as the artifact of the Experiment 279414948458319968
[7865:MainThread](2025-08-18 03:15:41,710) INFO - qlib.workflow - [record_temp.py:540] - Indicator analysis record 'indicator_analysis_1day.pkl' has been saved as the artifact of the Experiment 279414948458319968


'The following are analysis results of benchmark return(1day).'
                       risk
mean               0.000477
std                0.012295
annualized_return  0.113561
information_ratio  0.598699
max_drawdown      -0.370479
'The following are analysis results of the excess return without cost(1day).'
                       risk
mean               0.000730
std                0.005705
annualized_return  0.173708
information_ratio  1.973639
max_drawdown      -0.057265
'The following are analysis results of the excess return with cost(1day).'
                       risk
mean               0.000535
std                0.005703
annualized_return  0.127230
information_ratio  1.446068
max_drawdown      -0.066180
'The following are analysis results of indicators(1day).'
     value
ffr    1.0
pa     0.0
pos    0.0


[7865:MainThread](2025-08-18 03:15:43,563) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


# 分析图

In [ ]:
from qlib.contrib.report import analysis_model, analysis_position
from qlib.data import D

recorder = R.get_recorder(recorder_id=ba_rid, experiment_name="backtest_analysis")
print(recorder)
pred_df = recorder.load_object("pred.pkl")
report_normal_df = recorder.load_object("portfolio_analysis/report_normal_1day.pkl")
positions = recorder.load_object("portfolio_analysis/positions_normal_1day.pkl")
analysis_df = recorder.load_object("portfolio_analysis/port_analysis_1day.pkl")

{'class': 'Recorder', 'id': '13611d0507e34998a4a8054510429a96', 'name': 'mlflow_recorder', 'experiment_id': '279414948458319968', 'start_time': '2025-08-18 03:14:17', 'end_time': '2025-08-18 03:15:43', 'status': 'FINISHED'}


## 分析仓位

In [ ]:
analysis_position.report_graph(report_normal_df)

## 风险分析

In [ ]:
analysis_position.risk_analysis_graph(analysis_df, report_normal_df)

## 分析模型

In [ ]:
label_df = dataset.prepare("test", col_set="label")
label_df.columns = ["label"]

### IC得分

In [ ]:
pred_label = pd.concat([label_df, pred_df], axis=1, sort=True).reindex(label_df.index)
analysis_position.score_ic_graph(pred_label)

### 模型性能

In [ ]:
analysis_model.model_performance_graph(pred_label)